In [ ]:
import numpy as np
import torch


np.__version__

'2.0.2'

### Принципы авто. дифференцирования. (видео 1: скалярные функции)

##### Аналитическое решение

In [ ]:
"""    Аналитическое решение для частных производных    """

w_0 = np.array(29, dtype=np.float32)
w_1 = np.array(1.3, dtype=np.float32)
x = np.array([-9.5, 11], dtype=np.float32)
y = np.array([15, 52], dtype=np.float32)

# Прямой проход.
preds = w_0 + w_1*x   # (batch,)

# Подсчёт функции потерь MSE.
errors = y - preds   # (batch,)
train_loss = np.mean(errors**2)

# Обратный проход.
dw_0 = -2.0 * np.mean(errors)
dw_1 = -2.0 * np.mean(x * errors)

print(f"{dw_0 = }")
print(f"{dw_1 = }")

dw_0 = np.float32(-7.049999)
dw_1 = np.float32(-111.37503)


##### Reverse mode

In [ ]:
"""    Reverse mode    """

# Параметры модели.
w_0 = np.array(29, dtype=np.float32)
w_1 = np.array(1.3, dtype=np.float32)

# Данные первого наблюдения.
x1 = np.array(-9.5, dtype=np.float32)
y1 = np.array(15, dtype=np.float32)

# Данные второго наблюдения.
x2 = np.array(11, dtype=np.float32)
y2 = np.array(52, dtype=np.float32)

# Прямой проход (Forward)
c1 = x1 * w_1
c2 = x2 * w_1

k1 = c1 + w_0
k2 = c2 + w_0

# Подсчёт функции потерь MSE.
p1 = y1 - k1
p2 = y2 - k2

r1 = p1**2
r2 = p2**2

s = r1 + r2
L = s/2

# Обратный проход (Backward)
dL = 1
ds = dL * 1/2

dr1 = ds
dr2 = ds

dp2 = dr2 * 2 * p2
dp1 = dr1 * 2 * p1

dk2 = -dp2
dk1 = -dp1

dc2 = dk2
dc1 = dk1

dw_0 = dk2 + dk1
dw_1 = dc2 * x2 + dc1 * x1


print(f"d_c1 = {dc1}")
print(f"d_c2 = {dc2}\n")
print(f"d_k1 = {dk1}")
print(f"d_k2 = {dk2}\n")
print(f"d_p1 = {dp1}")
print(f"d_p2 = {dp2}\n")
print(f"d_r1 = {dr1}")
print(f"d_r2 = {dr2}\n")
print(f"d_s = {ds}")
print(f"d_L = {dL}\n\n")

print(f"{dw_0 = }")
print(f"{dw_1 = }")

d_c1 = 1.6500015258789062
d_c2 = -8.700000762939453

d_k1 = 1.6500015258789062
d_k2 = -8.700000762939453

d_p1 = -1.6500015258789062
d_p2 = 8.700000762939453

d_r1 = 0.5
d_r2 = 0.5

d_s = 0.5
d_L = 1


dw_0 = np.float32(-7.049999)
dw_1 = np.float32(-111.37503)


##### Авто. дифференцирование в PyTorch

In [ ]:
"""    Авто. дифференцирование в PyTorch (режим Reverse mode)    """

w_0 = torch.tensor(29, dtype=torch.float32, requires_grad=True)
w_1 = torch.tensor(1.3, dtype=torch.float32, requires_grad=True)
x = torch.tensor([-9.5, 11])
y = torch.tensor([15, 52])

# Прямой проход (Forward)
preds = w_0 + w_1*x

# Подсчёт функции потерь MSE.
errors = y - preds
train_loss = torch.mean(errors**2)

# Обратный проход (Backward)
train_loss.backward()

print(f"dw_0 = {w_0.grad}")
print(f"dw_1 = {w_1.grad}")

dw_0 = -7.049999237060547
dw_1 = -111.37503051757812


In [ ]:
# Параметры модели.
w_0 = torch.tensor(29, dtype=torch.float32, requires_grad=True)
w_1 = torch.tensor(1.3, dtype=torch.float32, requires_grad=True)

# Данные первого наблюдения.
x1 = torch.tensor(-9.5, dtype=torch.float32)
y1 = torch.tensor(15, dtype=torch.float32)

# Данные второго наблюдения.
x2 = torch.tensor(11, dtype=torch.float32)
y2 = torch.tensor(52, dtype=torch.float32)

# Прямой проход (Forward)
c1 = x1 * w_1
c2 = x2 * w_1

k1 = c1 + w_0
k2 = c2 + w_0

# Подсчёт функции потерь MSE.
p1 = y1 - k1
p2 = y2 - k2

r1 = p1**2
r2 = p2**2

s = r1 + r2
L = s/2

# Указываю Pytorch, что нужно сохранить градиенты для промежуточных значений.
c1.retain_grad()
c2.retain_grad()

k1.retain_grad()
k2.retain_grad()

p1.retain_grad()
p2.retain_grad()

r1.retain_grad()
r2.retain_grad()

s.retain_grad()
L.retain_grad()


# Обратный проход (Backward)
L.backward()

print(f"d_c1 = {c1.grad}")
print(f"d_c2 = {c2.grad}\n")
print(f"d_k1 = {k1.grad}")
print(f"d_k2 = {k2.grad}\n")
print(f"d_p1 = {p1.grad}")
print(f"d_p2 = {p2.grad}\n")
print(f"d_r1 = {r1.grad}")
print(f"d_r2 = {r2.grad}\n")
print(f"d_s = {s.grad}")
print(f"d_L = {L.grad}\n\n")

print(f"dw_0 = {w_0.grad}")
print(f"dw_1 = {w_1.grad}")

d_c1 = 1.6500015258789062
d_c2 = -8.700000762939453

d_k1 = 1.6500015258789062
d_k2 = -8.700000762939453

d_p1 = -1.6500015258789062
d_p2 = 8.700000762939453

d_r1 = 0.5
d_r2 = 0.5

d_s = 0.5
d_L = 1.0


dw_0 = -7.049999237060547
dw_1 = -111.37503051757812


### Принципы авто. дифференцирования. (видео 2: вектора и матрицы)

##### Reverse mode

In [ ]:
"""    Reverse mode    """

# Параметры модели.
W = np.ones((2, 3), dtype=np.float32)
b = np.ones(3, dtype=np.float32)

# Данные.
X = np.array([[1, 2], [2, 3]], dtype=np.float32)  # size(bs, 2)
Y = np.array([[2, 1, 3], [1, 2, 3]], dtype=np.float32)  # size(bs, 3)

# Прямой проход (Forward)
C = X @ W   # size(bs, 3)
Y_pred = C + b   # size(bs, 3)

# Подсчёт функции потерь MSE.
E = Y - Y_pred   # size(bs, 3)

R = E**2   # size(bs, 3)

s = np.sum(R)   # size(1,)
L = s / 6   # size(1,)

# Обратный проход (Backward)
dL = 1   # size(1,)
ds = dL * 1/6   # size(1,)

dR = ds * np.ones_like(R)   # size(bs, 3)

dE = dR * 2 * E   # size(bs, 3)

dY_pred = dE * (-1)   # size(bs, 3)

dC = dY_pred   # size(bs, 3)

db = np.sum(dY_pred, axis=0, keepdims=True)   # size(1, 3)
dW = X.T @ dC   # size(2, 3)


print(f"d_C =\n {dC.round(4)}\n")
print(f"d_Y_pred =\n {dY_pred.round(4)}\n")
print(f"d_E =\n {dE.round(4)}\n")
print(f"d_R =\n {dR.round(4)}\n")
print(f"d_s = {ds}")
print(f"d_L = {dL}\n\n")

print(f"d_W =\n{dW.round(4)}\n")
print(f"db = {db.round(4)}")

d_C =
 [[0.6667 1.     0.3333]
 [1.6667 1.3333 1.    ]]

d_Y_pred =
 [[0.6667 1.     0.3333]
 [1.6667 1.3333 1.    ]]

d_E =
 [[-0.6667 -1.     -0.3333]
 [-1.6667 -1.3333 -1.    ]]

d_R =
 [[0.1667 0.1667 0.1667]
 [0.1667 0.1667 0.1667]]

d_s = 0.16666666666666666
d_L = 1


d_W =
[[4.     3.6667 2.3333]
 [6.3333 6.     3.6667]]

db = [[2.3333 2.3333 1.3333]]


In [ ]:
"""    Авто. дифференцирование в PyTorch (режим Reverse mode)    """

# Параметры модели.
W = torch.ones((2, 3), dtype=torch.float32, requires_grad=True)
b = torch.ones(3, dtype=torch.float32, requires_grad=True)

# Данные.
X = torch.tensor([[1, 2], [2, 3]], dtype=torch.float32)
Y = torch.tensor([[2, 1, 3], [1, 2, 3]], dtype=torch.float32)

# Прямой проход (Forward)
C = X @ W
Y_pred = C + b

# Подсчёт функции потерь MSE.
E = Y - Y_pred

R = E**2

s = torch.sum(R)
L = s / 6

# Указываю Pytorch, что нужно сохранить градиенты для промежуточных значений.
C.retain_grad()
Y_pred.retain_grad()
E.retain_grad()
R.retain_grad()
s.retain_grad()
L.retain_grad()


# Обратный проход (Backward)
L.backward()

print(f"d_C =\n {C.grad}")
print(f"d_Y_pred =\n {Y_pred.grad}\n")
print(f"d_E =\n {E.grad}")
print(f"d_R =\n {R.grad}\n")
print(f"d_s = {s.grad}")
print(f"d_L = {L.grad}\n\n")

print(f"d_W =\n {W.grad}")
print(f"d_b =\n {b.grad}")

d_C =
 tensor([[0.6667, 1.0000, 0.3333],
        [1.6667, 1.3333, 1.0000]])
d_Y_pred =
 tensor([[0.6667, 1.0000, 0.3333],
        [1.6667, 1.3333, 1.0000]])

d_E =
 tensor([[-0.6667, -1.0000, -0.3333],
        [-1.6667, -1.3333, -1.0000]])
d_R =
 tensor([[0.1667, 0.1667, 0.1667],
        [0.1667, 0.1667, 0.1667]])

d_s = 0.1666666716337204
d_L = 1.0


d_W =
 tensor([[4.0000, 3.6667, 2.3333],
        [6.3333, 6.0000, 3.6667]])
d_b =
 tensor([2.3333, 2.3333, 1.3333])
